In [22]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

In [23]:
# 데이터 확인하기 2025.11.21
# 이상인 컬럼 제거 후 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler # 데이터 전처리용
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, recall_score, precision_score
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

import matplotlib.pyplot as plt
import seaborn as sns

import importlib
from utils import preprocessing

# 모듈 reload
importlib.reload(preprocessing)
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns
from utils.user_utils    import get_model_train_eval
from utils.model_utils   import save_model, load_model

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

# train = pd.read_csv("../data/train.csv")
# test  = pd.read_csv("../data/test.csv")

In [24]:
class ThresholdModel:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)


class ThresholdModel_rf:
    def __init__(self, base_model, threshold):
        self.base_model = base_model
        self.threshold = threshold

    def fit(self, X, y):
        return self

    def predict(self, X):
        proba = self.base_model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.base_model.predict_proba(X)


In [25]:
xgb = load_model("../models/XGB_100_HP_max5_est250_lr0.12_thr_0.15.pkl")
lgbm = load_model("../models/LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4_thr_0.44.pkl")
rf = load_model("../models/RF_100_HP_max7_est350_classWeight1vs4_thr_0.17.pkl")

✓ 모델 로드 완료: ../models\../models/XGB_100_HP_max5_est250_lr0.12_thr_0.15.pkl
  모델 타입: ThresholdModel
✓ 모델 로드 완료: ../models\../models/LGBM_100_HP_lr0.5_max6_min17_est350_num16_class1vs4_thr_0.44.pkl
  모델 타입: ThresholdModel
✓ 모델 로드 완료: ../models\../models/RF_100_HP_max7_est350_classWeight1vs4_thr_0.17.pkl
  모델 타입: ThresholdModel_rf


In [26]:
train, test = load_data()

X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=["ID"], axis=1)

# var3 처리
X_features["var3"] = X_features["var3"].replace(-999999, 2)

# train/val 분리
X_train, X_val, y_train, y_val = data_split(X_features, y_labels)


In [27]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [28]:
# ======================================
# 1) Base 모델들의 예측값 생성
# ======================================

xgb_proba = xgb.predict_proba(X_val_scaled)[:, 1]
lgbm_proba = lgbm.predict_proba(X_val_scaled)[:, 1]
rf_proba = rf.predict_proba(X_val)[:, 1]  # RF는 원본 사용해도 무방

# 스택킹용 입력 데이터 생성
stack_val = np.vstack([xgb_proba, lgbm_proba, rf_proba]).T

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [29]:
# RF 베이스 모델
rf_params={
 'class_weight': {0: 1, 1: 4},
 'criterion': 'gini',
 'max_depth': 7,
 'max_features': 1,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 3,
 'min_samples_split': 7,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 350,
}
rf_clf = RandomForestClassifier(
    **rf_params,
    random_state = 42,
    n_jobs      = -1
)

# XGB 베이스 모델
xgb_params = {
'learning_rate' : 0.12,
'max_depth': 5,
'subsample': 0.69
}
xgb_clf = XGBClassifier(
    objective       = "binary:logistic",
    eval_metric     = "logloss",
    tree_method     = "hist",
    n_jobs          = -1,
    random_state    = 42,
    **xgb_params
)

# LGBM 베이스 모델
lgbm_params = { 'learning_rate': 0.058,
 'max_depth': 6,
 'min_child_samples': 17,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 350,
 'num_leaves': 16,
 'class_weight': {0: 1, 1: 4}
 }
lgbm_clf = LGBMClassifier(
    objective   = "binary",
    n_jobs      = -1,
    random_state= 42,
    **lgbm_params
)

In [ ]:

# ======================================
# 1) 메타 모델 학습 (LogisticRegression)
# ======================================
meta_model = LogisticRegression(
    random_state = 23,
    max_iter     = 1000,
    C            = 0.029,
    penalty      = "l2",
    solver       = "lbfgs",
    class_weight = "balanced",
    n_jobs       = -1
)


# -------------------------------------------------
# 3) StackingClassifier 구성
#    - base estimators: RF, XGB, LGBM
#    - final_estimator: meta LR
#    - cv: StratifiedKFold(5)
# -------------------------------------------------
stacking_model = StackingClassifier(
    estimators      = [
        ("rf",   rf_clf),
        ("xgb",  xgb_clf),
        ("lgbm", lgbm_clf)
    ],
    final_estimator = meta_model,
    cv              = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method    = "predict_proba",
    n_jobs          = -1,
    passthrough     = False
)

print("\n" + "="*100)
print("Step. StackingClassifier 학습 (RF + XGB + LGBM → Meta LR)")
print("="*100)

# 학습: train 세트만 사용
stacking_model.fit(X_train_scaled, y_train)


Step. StackingClassifier 학습 (RF + XGB + LGBM → Meta LR)


,estimators,"[('rf', ...), ('xgb', ...), ...]"
,final_estimator,LogisticRegre...ndom_state=23)
,cv,StratifiedKFo... shuffle=True)
,stack_method,'predict_proba'
,n_jobs,-1
,passthrough,False
,verbose,0
,n_estimators,350
,criterion,'gini'
,max_depth,7
,min_samples_split,7


In [34]:
# ======================================
# 3) 스택킹 모델 성능 평가
# ======================================
# (1) 양성 클래스 확률
stack_proba_val = stacking_model.predict_proba(X_val_scaled)[:, 1]

# (2) 기본 threshold 0.5에서의 예측
stack_pred_val = (stack_proba_val >= 0.5).astype(int)

auc       = roc_auc_score(y_val, stack_proba_val)
acc       = accuracy_score(y_val, stack_pred_val)
precision = precision_score(y_val, stack_pred_val, zero_division=0)
recall    = recall_score(y_val, stack_pred_val, zero_division=0)
f1        = f1_score(y_val, stack_pred_val, zero_division=0)
cm        = confusion_matrix(y_val, stack_pred_val)

print("\n===== Stacking(LogReg on [RF, XGB, LGB]) 성능 (Threshold: 0.50) =====")
print(f"AUC       : {auc:.4f}")
print(f"정확도    : {acc:.4f}")
print(f"정밀도    : {precision:.4f}")
print(f"재현율    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_val, stack_pred_val, digits=4))



===== Stacking(LogReg on [RF, XGB, LGB]) 성능 (Threshold: 0.50) =====
AUC       : 0.8481
정확도    : 0.8411
정밀도    : 0.1543
재현율    : 0.6728
F1-score  : 0.2511

Confusion Matrix:
[[12383  2219]
 [  197   405]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9843    0.8480    0.9111     14602
           1     0.1543    0.6728    0.2511       602

    accuracy                         0.8411     15204
   macro avg     0.5693    0.7604    0.5811     15204
weighted avg     0.9515    0.8411    0.8850     15204



c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [36]:
# -----------------------------------------------
# 4) 메타 모델 기여도 출력 (StackingClassifier 버전)
# -----------------------------------------------
print("\n[4] 메타 모델 기여도 (Coefficients)")

# StackingClassifier 안에서 학습된 최종 메타 모델 (LogisticRegression)
meta_fitted = stacking_model.final_estimator_

# 각 base model의 양성클래스(1) 확률에 대한 계수
coef = meta_fitted.coef_[0]   # 길이 3: [RF, XGB, LGBM]

for name, c in zip(["RF", "XGB", "LGBM"], coef):
    print(f"{name} 기여도(계수): {c:.4f}")

# 선택: 인터셉트도 보고 싶으면
print(f"\nIntercept(절편): {meta_fitted.intercept_[0]:.4f}")

print("\n===== 스태킹 앙상블 완료 =====")


[4] 메타 모델 기여도 (Coefficients)
RF 기여도(계수): 0.9576
XGB 기여도(계수): 1.9456
LGBM 기여도(계수): 5.5153

Intercept(절편): -1.3363

===== 스태킹 앙상블 완료 =====


In [37]:
save_model(stacking_model, "Stacking_100_RF_XGB_LGBM_metaLR")

✓ 모델 저장 완료: ../models\Stacking_100_RF_XGB_LGBM_metaLR.pkl
  파일 크기: 1.73 MB


'../models\\Stacking_100_RF_XGB_LGBM_metaLR.pkl'

In [38]:
# ============================================================
# (추가) 로그 변환 + StackingClassifier + Threshold 최적화
# ============================================================

print("\n" + "="*100)
print("Step. 로그 변환 + StackingClassifier (RF + XGB + LGBM → Meta LR)")
print("="*100)

# 0) log1p 대상 컬럼 불러오기 ---------------------------------
log1p_cols_path = "../doc/Log1pColumns.txt"  # 🔸 실제 경로에 맞게 수정

with open(log1p_cols_path, encoding="utf-8") as f:
    log1p_cols = [c.strip() for c in f if c.strip()]

print(f"로그 변환 대상 컬럼 수: {len(log1p_cols)}")

# 1) 원본 feature 복사 후 log1p 적용 ---------------------------
X_features_log = X_features.copy()
X_test_log     = X_test.copy()

for col in log1p_cols:
    if col in X_features_log.columns:
        # 음수/0 방지용 clip, 필요 없으면 .clip 부분 제거해도 됨
        X_features_log[col] = np.log1p(X_features_log[col].clip(lower=0))
        X_test_log[col]     = np.log1p(X_test_log[col].clip(lower=0))

# var3 처리 (로그 버전에도 동일하게 적용)
X_features_log["var3"] = X_features_log["var3"].replace(-999999, 2)

# 2) train / val 분리 -----------------------------------------
X_train_log, X_val_log, y_train_log, y_val_log = data_split(X_features_log, y_labels)

# 3) 스케일링 -------------------------------------------------
scaler_log = StandardScaler()
X_train_log_scaled = scaler_log.fit_transform(X_train_log)
X_val_log_scaled   = scaler_log.transform(X_val_log)
X_test_log_scaled  = scaler_log.transform(X_test_log)

# 4) 같은 하이퍼파라미터로 베이스 모델 구성 -------------------
rf_clf_log = RandomForestClassifier(
    **rf_params,
    random_state = 42,
    n_jobs      = -1
)

xgb_clf_log = XGBClassifier(
    objective       = "binary:logistic",
    eval_metric     = "logloss",
    tree_method     = "hist",
    n_jobs          = -1,
    random_state    = 42,
    **xgb_params
)

lgbm_clf_log = LGBMClassifier(
    objective    = "binary",
    n_jobs       = -1,
    random_state = 42,
    **lgbm_params
)

# 5) 메타 모델 (동일 파라미터) --------------------------------
meta_model_log = LogisticRegression(
    random_state = 23,
    max_iter     = 1000,
    C            = 0.029,
    penalty      = "l2",
    solver       = "lbfgs",
    class_weight = "balanced",
    n_jobs       = -1
)

# 6) 로그 버전 StackingClassifier 구성 -------------------------
stacking_model_log = StackingClassifier(
    estimators      = [
        ("rf",   rf_clf_log),
        ("xgb",  xgb_clf_log),
        ("lgbm", lgbm_clf_log)
    ],
    final_estimator = meta_model_log,
    cv              = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method    = "predict_proba",
    n_jobs          = -1,
    passthrough     = False
)

# 7) 학습 ------------------------------------------------------
stacking_model_log.fit(X_train_log_scaled, y_train_log)

# 8) 기본 threshold=0.5 성능 평가 ------------------------------
stack_proba_val_log = stacking_model_log.predict_proba(X_val_log_scaled)[:, 1]
stack_pred_val_log  = (stack_proba_val_log >= 0.5).astype(int)

auc_log       = roc_auc_score(y_val_log, stack_proba_val_log)
acc_log       = accuracy_score(y_val_log, stack_pred_val_log)
precision_log = precision_score(y_val_log, stack_pred_val_log, zero_division=0)
recall_log    = recall_score(y_val_log, stack_pred_val_log, zero_division=0)
f1_log        = f1_score(y_val_log, stack_pred_val_log, zero_division=0)
cm_log        = confusion_matrix(y_val_log, stack_pred_val_log)

print("\n===== [LOG1P] Stacking(LogReg on [RF, XGB, LGB]) 성능 (Threshold: 0.50) =====")
print(f"AUC       : {auc_log:.4f}")
print(f"정확도    : {acc_log:.4f}")
print(f"정밀도    : {precision_log:.4f}")
print(f"재현율    : {recall_log:.4f}")
print(f"F1-score  : {f1_log:.4f}")
print("\nConfusion Matrix:")
print(cm_log)
print("\nClassification Report:")
print(classification_report(y_val_log, stack_pred_val_log, digits=4))

# 9) F1 기준 최적 threshold 탐색 ------------------------------
def find_best_threshold(y_true, proba, metric=f1_score, thr_list=None):
    if thr_list is None:
        thr_list = np.arange(0.05, 0.51, 0.01)
    best_thr   = 0.5
    best_score = -1
    best_rec   = 0.0

    for thr in thr_list:
        pred = (proba >= thr).astype(int)
        score = metric(y_true, pred, zero_division=0)
        rec   = recall_score(y_true, pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_thr   = thr
            best_rec   = rec
    return best_thr, best_score, best_rec

best_thr_log, best_f1_log, best_rec_log = find_best_threshold(y_val_log, stack_proba_val_log)

# 최적 threshold에서 다시 지표 계산
stack_pred_best_log = (stack_proba_val_log >= best_thr_log).astype(int)

auc_best_log       = roc_auc_score(y_val_log, stack_proba_val_log)
acc_best_log       = accuracy_score(y_val_log, stack_pred_best_log)
precision_best_log = precision_score(y_val_log, stack_pred_best_log, zero_division=0)
recall_best_log    = recall_score(y_val_log, stack_pred_best_log, zero_division=0)
f1_best_log        = f1_score(y_val_log, stack_pred_best_log, zero_division=0)
cm_best_log        = confusion_matrix(y_val_log, stack_pred_best_log)

print(f"\n===== [LOG1P] F1 기준 최적 Threshold 적용 성능 =====")
print(f"적용 Threshold : {best_thr_log:.2f}")
print(f"AUC (same proba): {auc_best_log:.4f}")
print(f"ACC : {acc_best_log:.4f}")
print(f"Precision : {precision_best_log:.4f}")
print(f"Recall    : {recall_best_log:.4f}")
print(f"F1        : {f1_best_log:.4f}")
print("\nConfusion Matrix:")
print(cm_best_log)
print("\nClassification Report:")
print(classification_report(y_val_log, stack_pred_best_log, digits=4))

# 10) 메타 모델 계수(기여도) 출력 -----------------------------
print("\n[LOG1P] 메타 모델 기여도 (Coefficients)")
meta_fitted_log = stacking_model_log.final_estimator_
coef_log = meta_fitted_log.coef_[0]

for name, c in zip(["RF", "XGB", "LGBM"], coef_log):
    print(f"{name} 기여도(계수): {c:.4f}")
print(f"\nIntercept(절편): {meta_fitted_log.intercept_[0]:.4f}")

print("\n===== [LOG1P] 스태킹 앙상블 완료 =====")

# (선택) 로그 버전 스태킹 전체 모델 저장
# save_model(stacking_model_log, "Stacking_log1p_RF_XGB_LGBM_metaLR")



Step. 로그 변환 + StackingClassifier (RF + XGB + LGBM → Meta LR)
로그 변환 대상 컬럼 수: 72


c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== [LOG1P] Stacking(LogReg on [RF, XGB, LGB]) 성능 (Threshold: 0.50) =====
AUC       : 0.8491
정확도    : 0.8408
정밀도    : 0.1548
재현율    : 0.6777
F1-score  : 0.2521

Confusion Matrix:
[[12375  2227]
 [  194   408]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9846    0.8475    0.9109     14602
           1     0.1548    0.6777    0.2521       602

    accuracy                         0.8408     15204
   macro avg     0.5697    0.7626    0.5815     15204
weighted avg     0.9517    0.8408    0.8848     15204


===== [LOG1P] F1 기준 최적 Threshold 적용 성능 =====
적용 Threshold : 0.50
AUC (same proba): 0.8491
ACC : 0.8408
Precision : 0.1548
Recall    : 0.6777
F1        : 0.2521

Confusion Matrix:
[[12375  2227]
 [  194   408]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9846    0.8475    0.9109     14602
           1     0.1548    0.6777    0.2521       602

    accuracy                         

In [39]:
save_model(stacking_model_log,'Stacking_100_log1p_RF_XGB_LGBM_metaLR')

✓ 모델 저장 완료: ../models\Stacking_100_log1p_RF_XGB_LGBM_metaLR.pkl
  파일 크기: 1.72 MB


'../models\\Stacking_100_log1p_RF_XGB_LGBM_metaLR.pkl'